# データ準備
## 前提
- プロジェクトルートで uv sync（取得はネット必須）
‐ raw / external は一度書いたら上書きしない（再取得は別ファイル名）
- ファイル名に 取得日 YYYYMMDD を入れる（data-catalog）
- pybaseball は pandas を返す → 保存・分析は polars に載せ替える

In [ ]:
# 共通セットアップ
from datetime import date

import polars as pl
from pybaseball import chadwick_register, pitching_stats, playerid_lookup, statcast

from analysis_project.paths import data_dir, ensure_parent_dir

FETCH_DATE = date.today().strftime("%Y%m%d")  # 例: 20260921


# 山本由伸の ID 固定（マイルストーン1）
## Keys
- key_mlbam=808967
- key_fangraphs=33825

In [9]:
ids = playerid_lookup("yamamoto", "yoshinobu")
print(ids)


# 山本由伸の ID 固定（マイルストーン2）
## 例（2026-03 時点の lookup 結果）:
## key_mlbam=808967, key_fangraphs=33825

YAMAMOTO_MLBAM = 808967
YAMAMOTO_FANG = 33825

# データ取得
register_dir = data_dir() / "external" / "register"

# parquetで保存する
output_path = ensure_parent_dir(register_dir / f"{FETCH_DATE}_yamamoto_ids.parquet")
pl.from_pandas(ids).write_parquet(output_path)

# Register 全体（初回のみ　結合用）
reg_path = ensure_parent_dir(register_dir / f"{FETCH_DATE}_chadwick_register.parquet")

if not reg_path.exists(): # external も「上書きしない」運用なら別日付で新規保存
    reg = chadwick_register()
    pl.from_pandas(reg).write_parquet(reg_path)


  name_last name_first  key_mlbam key_retro  key_bbref  key_fangraphs  \
0  yamamoto  yoshinobu     808967  yamay001  yamamyo01          33825   

   mlb_played_first  mlb_played_last  
0            2024.0           2026.0  
Gathering player lookup table. This may take a moment.
